<a href="https://colab.research.google.com/github/varakalicheva-hub/diplom_2026/blob/main/%D0%9A%D0%BE%D0%B4_%D0%B4%D0%BB%D1%8F_%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B8_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%BE_%D0%BF%D0%BE_%D1%81%D0%B5%D0%BC%D0%B0%D0%BD%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B9_%D0%B1%D0%BB%D0%B8%D0%B7%D0%BE%D1%81%D1%82%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
print(sys.executable)

try:
    from sentence_transformers import SentenceTransformer
    print("sentence_transformers импортируется успешно")
except ImportError as e:
    print("Ошибка импорта:", e)

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Загрузка модели FRIDA...")
model = SentenceTransformer('ai-forever/FRIDA')
print("Модель загружена.")


def compute_similarity(text1, text2):
    if pd.isna(text1) or pd.isna(text2) or text1 == "" or text2 == "":
        return 0.0
    emb1 = model.encode([text1])[0]
    emb2 = model.encode([text2])[0]
    sim = cosine_similarity([emb1], [emb2])[0][0]
    return sim


def semantic_evaluation(golden_df, pred_df):
    # сравниваем датасеты по косинусному сходству эмбеддингов (FRIDA).
    # жадное сопоставление: для каждого gold-квада ищем лучший pred с тем же аспектом+тональностью,
    # остальное попадает в FN (пропущенные) или FP (лишние)

    golden_df = golden_df.copy()
    pred_df   = pred_df.copy()
    golden_df['review_id'] = golden_df['review_id'].astype(str)
    pred_df['review_id']   = pred_df['review_id'].astype(str)

    golden_grouped = golden_df.groupby('review_id')
    pred_grouped   = pred_df.groupby('review_id')

    all_review_ids = set(golden_grouped.groups.keys()) | set(pred_grouped.groups.keys())
    print(f"Найдено {len(all_review_ids)} review_id для сравнения.")

    results = []

    for rid in tqdm(all_review_ids, desc="Обработка отзывов"):
        gold_rows = (
            golden_grouped.get_group(rid)
            if rid in golden_grouped.groups
            else pd.DataFrame()
        )
        pred_rows = (
            pred_grouped.get_group(rid)
            if rid in pred_grouped.groups
            else pd.DataFrame()
        )

        def to_quads(rows):
            quads = []
            for _, row in rows.iterrows():
                quads.append({
                    'aspect_term': row['aspect_term'],
                    'opinion':     row['opinion'],
                    'aspect':      str(row['aspect']).strip().lower(),
                    'sentiment':   str(row['sentiment']).strip().lower(),
                    'aspect_term_orig': row['aspect_term'],
                    'opinion_orig':     row['opinion'],
                })
            return quads

        gold_quads = to_quads(gold_rows)
        pred_quads = to_quads(pred_rows)

        # жадное сопоставление: идём по gold, для каждого ищем лучший pred с тем же aspect+sentiment
        matched_pred_indices = set()
        matched_gold_indices = set()

        for gi, gq in enumerate(gold_quads):
            best_j     = -1
            best_score = -1.0
            for ji, pq in enumerate(pred_quads):
                if ji in matched_pred_indices:
                    continue
                if gq['aspect'] != pq['aspect'] or gq['sentiment'] != pq['sentiment']:
                    continue
                term_sim = compute_similarity(gq['aspect_term'], pq['aspect_term'])
                op_sim   = compute_similarity(gq['opinion'],     pq['opinion'])
                score    = (term_sim + op_sim) / 2
                if score > best_score:
                    best_score = score
                    best_j     = ji

            if best_j != -1:
                pq = pred_quads[best_j]
                matched_pred_indices.add(best_j)
                matched_gold_indices.add(gi)
                term_sim = compute_similarity(gq['aspect_term'], pq['aspect_term'])
                op_sim   = compute_similarity(gq['opinion'],     pq['opinion'])
                results.append({
                    'review_id':              rid,
                    'match_type':             'matched',
                    'gold_aspect_term':       gq['aspect_term_orig'],
                    'pred_aspect_term':       pq['aspect_term_orig'],
                    'aspect_term_similarity': term_sim,
                    'gold_opinion':           gq['opinion_orig'],
                    'pred_opinion':           pq['opinion_orig'],
                    'opinion_similarity':     op_sim,
                    'aspect':                 gq['aspect'],
                    'sentiment':              gq['sentiment'],
                    'avg_similarity':         (term_sim + op_sim) / 2,
                })

        # пропущенные: есть в золотом, но не нашлось пары в предсказанном
        for gi, gq in enumerate(gold_quads):
            if gi in matched_gold_indices:
                continue
            results.append({
                'review_id':              rid,
                'match_type':             'missed (FN)',
                'gold_aspect_term':       gq['aspect_term_orig'],
                'pred_aspect_term':       None,
                'aspect_term_similarity': 0.0,
                'gold_opinion':           gq['opinion_orig'],
                'pred_opinion':           None,
                'opinion_similarity':     0.0,
                'aspect':                 gq['aspect'],
                'sentiment':              gq['sentiment'],
                'avg_similarity':         0.0,
            })

        # лишние: есть в предсказанном, но не взяли их ни под один gold
        for ji, pq in enumerate(pred_quads):
            if ji in matched_pred_indices:
                continue
            results.append({
                'review_id':              rid,
                'match_type':             'extra (FP)',
                'gold_aspect_term':       None,
                'pred_aspect_term':       pq['aspect_term_orig'],
                'aspect_term_similarity': 0.0,
                'gold_opinion':           None,
                'pred_opinion':           pq['opinion_orig'],
                'opinion_similarity':     0.0,
                'aspect':                 pq['aspect'],
                'sentiment':              pq['sentiment'],
                'avg_similarity':         0.0,
            })

    results_df = pd.DataFrame(results)

    # статистика считается только по matched-парам (FN и FP дают 0.0, их не мешаем)
    if not results_df.empty:
        matched = results_df[results_df['match_type'] == 'matched']
        summary = {
            'total_quads':          len(results_df),
            'matched_pairs':        len(matched),
            'missed_fn':            (results_df['match_type'] == 'missed (FN)').sum(),
            'extra_fp':             (results_df['match_type'] == 'extra (FP)').sum(),

            'aspect_term_similarity_mean':   matched['aspect_term_similarity'].mean(),
            'opinion_similarity_mean':       matched['opinion_similarity'].mean(),
            'avg_similarity_mean':           matched['avg_similarity'].mean(),

            'aspect_term_similarity_median': matched['aspect_term_similarity'].median(),
            'opinion_similarity_median':     matched['opinion_similarity'].median(),
            'avg_similarity_median':         matched['avg_similarity'].median(),

            'aspect_term_similarity_std':    matched['aspect_term_similarity'].std(),
            'opinion_similarity_std':        matched['opinion_similarity'].std(),
            'avg_similarity_std':            matched['avg_similarity'].std(),

            'aspect_term_q25': matched['aspect_term_similarity'].quantile(0.25),
            'aspect_term_q75': matched['aspect_term_similarity'].quantile(0.75),
            'opinion_q25':     matched['opinion_similarity'].quantile(0.25),
            'opinion_q75':     matched['opinion_similarity'].quantile(0.75),
            'avg_q25':         matched['avg_similarity'].quantile(0.25),
            'avg_q75':         matched['avg_similarity'].quantile(0.75),
        }
    else:
        summary = {'error': 'No pairs found'}

    return results_df, summary


if __name__ == "__main__":
    golden_df = pd.read_csv('golden_dataset_3.csv')
    pred_df   = pd.read_csv('predictions_dataset_deepseek.csv')

    results_df, summary = semantic_evaluation(golden_df, pred_df)

    print("\n" + "=" * 60)
    print("         СВОДНАЯ СТАТИСТИКА СЕМАНТИЧЕСКОЙ БЛИЗОСТИ")
    print("=" * 60)
    print(f"Всего квадруплетов обработано : {summary.get('total_quads', 0)}")
    print(f"  Сопоставленные пары       : {summary.get('matched_pairs', 0)}")
    print(f"  Пропущенные (FN)          : {summary.get('missed_fn', 0)}")
    print(f"  Лишние (FP)               : {summary.get('extra_fp', 0)}")

    print("\n--- СРЕДНИЕ ЗНАЧЕНИЯ (по сопоставленным парам) ---")
    print(f"Среднее сходство аспектных терминов : {summary.get('aspect_term_similarity_mean', 0):.4f}")
    print(f"Среднее сходство мнений             : {summary.get('opinion_similarity_mean', 0):.4f}")
    print(f"Среднее комбинированное сходство    : {summary.get('avg_similarity_mean', 0):.4f}")

    print("\n--- МЕДИАНЫ ---")
    print(f"Медиана аспектных терминов : {summary.get('aspect_term_similarity_median', 0):.4f}")
    print(f"Медиана мнений             : {summary.get('opinion_similarity_median', 0):.4f}")
    print(f"Медиана комбинированного   : {summary.get('avg_similarity_median', 0):.4f}")

    print("\n--- СТАНДАРТНЫЕ ОТКЛОНЕНИЯ ---")
    print(f"Стд. откл. аспектных терминов : {summary.get('aspect_term_similarity_std', 0):.4f}")
    print(f"Стд. откл. мнений             : {summary.get('opinion_similarity_std', 0):.4f}")
    print(f"Стд. откл. комбинированного   : {summary.get('avg_similarity_std', 0):.4f}")

    results_df.to_csv('semantic_similarity_results.csv', index=False, encoding='utf-8-sig')
    print("\nДетальные результаты сохранены в 'semantic_similarity_results.csv'")
    print("Колонка 'match_type': matched | missed (FN) | extra (FP)")